# FEATURE ENGINEERING V2

# 0. Setup & Load Data

In [3]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

Saving nba_full_dataset (1).csv to nba_full_dataset (1) (1).csv
User uploaded file "nba_full_dataset (1) (1).csv" with length 2245140 bytes


In [4]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("nba_full_dataset (1).csv")
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

print(f'Dataset gốc: {df.shape[0]} dòng × {df.shape[1]} cột')
print(f'Số trận: {df["GAME_ID"].nunique()}')
print(f'Mùa giải: {sorted(df["SEASON"].unique())}')
print(f'\nCác cột hiện có:')
print(list(df.columns))

Dataset gốc: 12300 dòng × 36 cột
Số trận: 6150
Mùa giải: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

Các cột hiện có:
['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON', 'IS_HOME', 'WIN', 'REST_DAYS', 'IS_B2B', 'GAMES_PLAYED_SEASON', 'CURRENT_WIN_PCT', 'WIN_STREAK']


# 1. Xử lý vấn đề phát hiện từ EDA (giống V1)

In [5]:
# 1.1 — Fill missing
df['FT_PCT'] = df['FT_PCT'].fillna(0)

# 1.2 — Cap REST_DAYS
print(f'REST_DAYS trước khi cap: min={df["REST_DAYS"].min()}, max={df["REST_DAYS"].max()}')
df['REST_DAYS'] = df['REST_DAYS'].clip(upper=10)
print(f'REST_DAYS sau khi cap:   min={df["REST_DAYS"].min()}, max={df["REST_DAYS"].max()}')

# 1.3 — Loại trận neutral site
game_home_count = df.groupby('GAME_ID')['IS_HOME'].sum()
neutral_games = game_home_count[game_home_count != 1].index
n_neutral = len(neutral_games)
df = df[~df['GAME_ID'].isin(neutral_games)].copy()

print(f'\nLoại {n_neutral} trận neutral site')
print(f'Dataset sau xử lý: {df.shape[0]} dòng, {df["GAME_ID"].nunique()} trận')

# Xác nhận
home_count = (df['IS_HOME'] == 1).sum()
away_count = (df['IS_HOME'] == 0).sum()
print(f'HOME: {home_count}, AWAY: {away_count} → {"Bằng nhau" if home_count == away_count else "❌ Lệch!"}')

REST_DAYS trước khi cap: min=1.0, max=9.0
REST_DAYS sau khi cap:   min=1.0, max=9.0

Loại 10 trận neutral site
Dataset sau xử lý: 12280 dòng, 6140 trận
HOME: 6140, AWAY: 6140 → Bằng nhau


# 2. (MỚI) Elo Rating


In [ ]:
K = 20
HOME_ADVANTAGE = 100
INITIAL_ELO = 1500
SEASON_CARRYOVER = 0.75  # 75% Elo cũ + 25% baseline

# Bước 2.1: Tạo danh sách trận (1 dòng/trận) để xử lý tuần tự
home_rows = df[df['IS_HOME'] == 1][['GAME_ID', 'GAME_DATE', 'SEASON', 'TEAM_ID', 'WIN']].copy()
home_rows.columns = ['GAME_ID', 'GAME_DATE', 'SEASON', 'HOME_TEAM_ID', 'HOME_WIN']
away_rows = df[df['IS_HOME'] == 0][['GAME_ID', 'TEAM_ID']].copy()
away_rows.columns = ['GAME_ID', 'AWAY_TEAM_ID']

games = home_rows.merge(away_rows, on='GAME_ID').sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
print(f'Danh sách trận: {len(games)} trận, sắp xếp theo thời gian')

# Bước 2.2: Tính Elo tuần tự
elo_ratings = {}  # TEAM_ID -> current Elo
game_elo = {}     # GAME_ID -> (home_elo_before, away_elo_before)
processed_seasons = set()

for _, game in games.iterrows():
    season = game['SEASON']
    home_id = game['HOME_TEAM_ID']
    away_id = game['AWAY_TEAM_ID']

    # Season reset
    if season not in processed_seasons:
        processed_seasons.add(season)
        if elo_ratings:
            for tid in elo_ratings:
                elo_ratings[tid] = SEASON_CARRYOVER * elo_ratings[tid] + (1 - SEASON_CARRYOVER) * INITIAL_ELO
            print(f'  Season reset → {season}: avg Elo = {np.mean(list(elo_ratings.values())):.1f}')

    # Khởi tạo
    if home_id not in elo_ratings:
        elo_ratings[home_id] = INITIAL_ELO
    if away_id not in elo_ratings:
        elo_ratings[away_id] = INITIAL_ELO

    R_home = elo_ratings[home_id]
    R_away = elo_ratings[away_id]

    # Lưu Elo TRƯỚC trận
    game_elo[game['GAME_ID']] = (R_home, R_away)

    # Expected (có home advantage +100)
    E_home = 1 / (1 + 10 ** ((R_away - (R_home + HOME_ADVANTAGE)) / 400))
    E_away = 1 - E_home

    # Update
    S_home = float(game['HOME_WIN'])
    S_away = 1.0 - S_home
    elo_ratings[home_id] = R_home + K * (S_home - E_home)
    elo_ratings[away_id] = R_away + K * (S_away - E_away)

# Bước 2.3: Map Elo về df (mỗi dòng = 1 đội trong 1 trận)
def get_elo(row):
    gid = row['GAME_ID']
    if gid in game_elo:
        home_elo, away_elo = game_elo[gid]
        return home_elo if row['IS_HOME'] == 1 else away_elo
    return INITIAL_ELO

df['ELO'] = df.apply(get_elo, axis=1)

print(f'\nĐã tính Elo cho {len(game_elo)} trận')
print(f'Elo range: {df["ELO"].min():.0f} → {df["ELO"].max():.0f}')
print(f'Elo mean: {df["ELO"].mean():.0f}')

Danh sách trận: 6140 trận, sắp xếp theo thời gian
  Season reset → 2022-23: avg Elo = 1500.0
  Season reset → 2023-24: avg Elo = 1500.0
  Season reset → 2024-25: avg Elo = 1500.0
  Season reset → 2025-26: avg Elo = 1500.0

Đã tính Elo cho 6140 trận
Elo range: 1201 → 1781
Elo mean: 1500


In [7]:
# Kiểm tra Elo — Top/Bottom đội cuối mỗi mùa
print('Top 5 Elo cao nhất cuối mỗi mùa:')
print('=' * 60)

for season in sorted(df['SEASON'].unique()):
    season_df = df[df['SEASON'] == season]
    # Lấy dòng cuối cùng của mỗi đội trong mùa
    last_game = season_df.sort_values('GAME_DATE').groupby('TEAM_ABBREVIATION').last()
    top5 = last_game.nlargest(5, 'ELO')[['ELO']]
    teams = ', '.join([f'{t}({e:.0f})' for t, e in zip(top5.index, top5['ELO'])])
    print(f'  {season}: {teams}')

Top 5 Elo cao nhất cuối mỗi mùa:
  2021-22: PHX(1697), MEM(1646), DAL(1624), BOS(1616), MIL(1611)
  2022-23: MIL(1672), BOS(1640), PHI(1636), CLE(1602), MEM(1596)
  2023-24: BOS(1707), DEN(1640), OKC(1634), MIN(1630), DAL(1610)
  2024-25: OKC(1759), BOS(1700), CLE(1688), LAC(1623), MIN(1610)
  2025-26: OKC(1761), SAS(1718), BOS(1674), DET(1658), NYK(1635)


# 3. (MỚI) - Oliver's Four Factors (3/4)

In [ ]:
# 3.1 — eFG%: (FGM + 0.5 * FG3M) / FGA
df['eFG_PCT'] = (df['FGM'] + 0.5 * df['FG3M']) / df['FGA']

# 3.2 — Possessions (dùng để tính TO Ratio)
df['POSS'] = df['FGA'] + 0.44 * df['FTA'] - df['OREB'] + df['TOV']

# 3.3 — TO Ratio: TOV / Poss
df['TO_RATIO'] = df['TOV'] / df['POSS']
# Xử lý edge case: POSS = 0 (cực kỳ hiếm)
df['TO_RATIO'] = df['TO_RATIO'].fillna(0)

# 3.4 — FT Rate: FTM / FGA
df['FT_RATE'] = df['FTM'] / df['FGA']
df['FT_RATE'] = df['FT_RATE'].fillna(0)

print('Oliver\'s Four Factors — Thống kê raw:')
print('=' * 50)
for col in ['eFG_PCT', 'TO_RATIO', 'FT_RATE']:
    print(f'  {col:12s}: mean={df[col].mean():.4f}, std={df[col].std():.4f}, '
          f'min={df[col].min():.4f}, max={df[col].max():.4f}')

print(f'\nPOSS (possession estimate): mean={df["POSS"].mean():.1f}, std={df["POSS"].std():.1f}')

Oliver's Four Factors — Thống kê raw:
  eFG_PCT     : mean=0.5437, std=0.0661, min=0.3193, max=0.8077
  TO_RATIO    : mean=0.1316, std=0.0367, min=0.0099, max=0.2969
  FT_RATE     : mean=0.2001, std=0.0730, min=0.0000, max=0.5789

POSS (possession estimate): mean=101.2, std=5.6


# 4. EMA Rolling Average (V1 + Four Factors mới)

In [9]:
EMA_FEATURES_V1 = ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                   'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV']

EMA_FEATURES_NEW = ['eFG_PCT', 'TO_RATIO', 'FT_RATE']

EMA_FEATURES_ALL = EMA_FEATURES_V1 + EMA_FEATURES_NEW

EMA_SPAN = 5

print(f'EMA span: {EMA_SPAN}')
print(f'V1 features ({len(EMA_FEATURES_V1)}): {EMA_FEATURES_V1}')
print(f'New features ({len(EMA_FEATURES_NEW)}): {EMA_FEATURES_NEW}')
print(f'Total: {len(EMA_FEATURES_ALL)} features')

EMA span: 5
V1 features (10): ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV']
New features (3): ['eFG_PCT', 'TO_RATIO', 'FT_RATE']
Total: 13 features


In [10]:
# Sắp xếp theo đội + thời gian
df = df.sort_values(['TEAM_ID', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

# Tính EMA cho tất cả features
for feat in EMA_FEATURES_ALL:
    col_name = f'EMA_{feat}'
    # shift(1): không bao gồm trận hiện tại
    shifted = df.groupby(['TEAM_ID', 'SEASON'])[feat].shift(1)
    # EMA span=5
    df[col_name] = shifted.groupby([df['TEAM_ID'], df['SEASON']]).apply(
        lambda x: x.ewm(span=EMA_SPAN, min_periods=1).mean()
    ).reset_index(level=[0, 1], drop=True)

# Kiểm tra
print('Ví dụ: 8 trận đầu của BOS mùa 2023-24')
print('-' * 100)
bos_mask = (df['TEAM_ABBREVIATION'] == 'BOS') & (df['SEASON'] == '2023-24')
check_cols = ['GAME_DATE', 'PTS', 'EMA_PTS', 'eFG_PCT', 'EMA_eFG_PCT', 'TO_RATIO', 'EMA_TO_RATIO']
print(df[bos_mask].head(8)[check_cols].to_string(index=False))

print(f'\n→ Trận 1: NaN (chưa có lịch sử — shift(1))')
print(f'→ Trận 2+: EMA bắt đầu tích lũy')

Ví dụ: 8 trận đầu của BOS mùa 2023-24
----------------------------------------------------------------------------------------------------
 GAME_DATE  PTS    EMA_PTS  eFG_PCT  EMA_eFG_PCT  TO_RATIO  EMA_TO_RATIO
2023-10-25  108        NaN 0.558442          NaN  0.137654           NaN
2023-10-27  119 108.000000 0.557895     0.558442  0.146542      0.137654
2023-10-30  126 114.600000 0.593137     0.558113  0.158760      0.142986
2023-11-01  155 120.000000 0.673684     0.574704  0.102497      0.150458
2023-11-04  124 134.538462 0.561111     0.615819  0.106921      0.130536
2023-11-06  109 130.492891 0.451087     0.594817  0.142908      0.121470
2023-11-08  103 122.639098 0.478022     0.542296  0.117233      0.129304
2023-11-10  121 115.685770 0.537234     0.519540  0.072674      0.125030

→ Trận 1: NaN (chưa có lịch sử — shift(1))
→ Trận 2+: EMA bắt đầu tích lũy


In [11]:
# Kiểm tra NaN
ema_cols = [f'EMA_{f}' for f in EMA_FEATURES_ALL]
nan_rows = df[ema_cols].isnull().any(axis=1).sum()
print(f'Số dòng NaN ở EMA: {nan_rows} (trận đầu mỗi mùa mỗi đội)')
print(f'Chiếm {nan_rows/len(df)*100:.1f}% — sẽ bị loại khi ghép')

Số dòng NaN ở EMA: 150 (trận đầu mỗi mùa mỗi đội)
Chiếm 1.2% — sẽ bị loại khi ghép


# 5. Ghép 2 đội thành 1 dòng/trận

Giống V1 nhưng thêm ELO + 3 EMA Four Factors mới

In [12]:
team_features = (
    [f'EMA_{f}' for f in EMA_FEATURES_ALL]   # 13 EMA features (bao gồm EMA_PTS)
    + ['CURRENT_WIN_PCT', 'WIN_STREAK', 'REST_DAYS', 'IS_B2B', 'GAMES_PLAYED_SEASON']
    + ['ELO']
)
game_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']

# Tách HOME / AWAY, thêm cột PTS thực tế
home_df = df[df['IS_HOME'] == 1].copy()
away_df = df[df['IS_HOME'] == 0].copy()

# Rename: thêm HOME_ và AWAY_, bao gồm PTS
home_rename = {col: f'HOME_{col}' for col in team_features}
home_rename['TEAM_ABBREVIATION'] = 'HOME_TEAM'
home_rename['WIN'] = 'HOME_WIN'
home_rename['PTS'] = 'HOME_PTS'          # THÊM: điểm thực tế của đội nhà

away_rename = {col: f'AWAY_{col}' for col in team_features}
away_rename['TEAM_ABBREVIATION'] = 'AWAY_TEAM'
away_rename['PTS'] = 'AWAY_PTS'          # THÊM: điểm thực tế của đội khách

home_df = home_df.rename(columns=home_rename)
away_df = away_df.rename(columns=away_rename)

# Chọn cột để merge
home_cols = game_cols + ['HOME_TEAM', 'HOME_WIN', 'HOME_PTS'] + [f'HOME_{f}' for f in team_features]
away_cols = ['GAME_ID', 'AWAY_TEAM', 'AWAY_PTS'] + [f'AWAY_{f}' for f in team_features]

merged = home_df[home_cols].merge(away_df[away_cols], on='GAME_ID', how='inner')

# Loại NaN
before = len(merged)
merged = merged.dropna().reset_index(drop=True)
print(f'Sau ghép: {before} → loại {before - len(merged)} NaN → còn {len(merged)} trận')
print(f'Shape: {merged.shape}')

# ----------------------------------------------
# BỔ SUNG CÁC CỘT CHO REGRESSION
# ----------------------------------------------
# MARGIN = HOME_PTS - AWAY_PTS (target cho regression)
merged['MARGIN'] = merged['HOME_PTS'] - merged['AWAY_PTS']

# Điểm cho phép (points allowed) dùng EMA_PTS của đối thủ
merged['HOME_EMA_PTS_ALLOWED'] = merged['AWAY_EMA_PTS']
merged['AWAY_EMA_PTS_ALLOWED'] = merged['HOME_EMA_PTS']
merged['DIFF_PTS_ALLOWED'] = merged['HOME_EMA_PTS_ALLOWED'] - merged['AWAY_EMA_PTS_ALLOWED']

print("Đã thêm MARGIN (target) và DIFF_PTS_ALLOWED (feature mới)")

Sau ghép: 6140 → loại 79 NaN → còn 6061 trận
Shape: (6061, 46)
Đã thêm MARGIN (target) và DIFF_PTS_ALLOWED (feature mới)


In [13]:
# OREB% — tính sau ghép vì cần Opp_DREB
merged['HOME_OREB_PCT'] = merged['HOME_EMA_OREB'] / (merged['HOME_EMA_OREB'] + merged['AWAY_EMA_DREB'])
merged['AWAY_OREB_PCT'] = merged['AWAY_EMA_OREB'] / (merged['AWAY_EMA_DREB'] + merged['HOME_EMA_DREB'])
merged['HOME_OREB_PCT'] = merged['HOME_OREB_PCT'].fillna(0)
merged['AWAY_OREB_PCT'] = merged['AWAY_OREB_PCT'].fillna(0)


# V1 DIFF features (giữ nguyên)
DIFF_FEATURES_V1 = {
    'EMA_PTS': 'DIFF_PTS',
    'EMA_FG_PCT': 'DIFF_FG_PCT',
    'EMA_FG3_PCT': 'DIFF_FG3_PCT',
    'EMA_FT_PCT': 'DIFF_FT_PCT',
    'EMA_OREB': 'DIFF_OREB',
    'EMA_DREB': 'DIFF_DREB',
    'EMA_AST': 'DIFF_AST',
    'EMA_STL': 'DIFF_STL',
    'EMA_BLK': 'DIFF_BLK',
    'EMA_TOV': 'DIFF_TOV',
    'CURRENT_WIN_PCT': 'DIFF_WIN_PCT',
    'WIN_STREAK': 'DIFF_WIN_STREAK',
    'REST_DAYS': 'DIFF_REST_DAYS',
}


DIFF_FEATURES_V2 = {
    'ELO': 'DIFF_ELO',
    'EMA_eFG_PCT': 'DIFF_eFG_PCT',
    'EMA_TO_RATIO': 'DIFF_TO_RATIO',
    'EMA_FT_RATE': 'DIFF_FT_RATE',
    'OREB_PCT': 'DIFF_OREB_PCT',
}

# Tất cả DIFF
DIFF_FEATURES_ALL = {**DIFF_FEATURES_V1, **DIFF_FEATURES_V2}

for feat, diff_name in DIFF_FEATURES_ALL.items():
    merged[diff_name] = merged[f'HOME_{feat}'] - merged[f'AWAY_{feat}']

print(f'Đã tạo {len(DIFF_FEATURES_ALL)} difference features:')
print(f'  V1 (giữ nguyên): {len(DIFF_FEATURES_V1)}')
print(f'  V2 (mới):        {len(DIFF_FEATURES_V2)}')
print()

# Thống kê chi tiết
print(f'{"Feature":22s} {"Mean":>8s} {"Std":>8s}')
print('-' * 40)
for diff_name in DIFF_FEATURES_ALL.values():
    val = merged[diff_name]
    marker = ' ⭐' if diff_name in DIFF_FEATURES_V2.values() else ''
    print(f'{diff_name:22s} {val.mean():+8.3f} {val.std():8.3f}{marker}')

Đã tạo 18 difference features:
  V1 (giữ nguyên): 13
  V2 (mới):        5

Feature                    Mean      Std
----------------------------------------
DIFF_PTS                 -0.206    9.488
DIFF_FG_PCT              -0.000    0.041
DIFF_FG3_PCT             -0.002    0.057
DIFF_FT_PCT              -0.000    0.074
DIFF_OREB                +0.013    3.130
DIFF_DREB                +0.021    4.071
DIFF_AST                 -0.055    4.142
DIFF_STL                 -0.014    2.220
DIFF_BLK                 -0.015    1.887
DIFF_TOV                 -0.002    2.827
DIFF_WIN_PCT             -0.003    0.261
DIFF_WIN_STREAK          +0.054    4.746
DIFF_REST_DAYS           +0.081    1.000
DIFF_ELO                 +0.963  139.052 ⭐
DIFF_eFG_PCT             -0.001    0.049 ⭐
DIFF_TO_RATIO            +0.000    0.027 ⭐
DIFF_FT_RATE             +0.001    0.052 ⭐
DIFF_OREB_PCT            +0.081    0.052 ⭐


# 7. Định nghĩa bộ Features V3

In [ ]:
# Bộ V2 cũ (20 features)
feature_set_v2 = list(DIFF_FEATURES_V1.values()) + ['HOME_IS_B2B', 'AWAY_IS_B2B'] + list(DIFF_FEATURES_V2.values())

# Bộ V3: thêm DIFF_PTS_ALLOWED (tổng 21 features)
feature_set_v3 = feature_set_v2 + ['DIFF_PTS_ALLOWED']

print(f'Bộ V2: {len(feature_set_v2)} features')
print(f'Bộ V3: {len(feature_set_v3)} features')
print("\nFeatures V3 (21 cột):")
for i, f in enumerate(feature_set_v3, 1):
    marker = ' ⭐ MỚI' if f == 'DIFF_PTS_ALLOWED' else ''
    print(f'  {i:2d}. {f}{marker}')

Bộ V2: 20 features
Bộ V3: 21 features

Features V3 (21 cột):
   1. DIFF_PTS
   2. DIFF_FG_PCT
   3. DIFF_FG3_PCT
   4. DIFF_FT_PCT
   5. DIFF_OREB
   6. DIFF_DREB
   7. DIFF_AST
   8. DIFF_STL
   9. DIFF_BLK
  10. DIFF_TOV
  11. DIFF_WIN_PCT
  12. DIFF_WIN_STREAK
  13. DIFF_REST_DAYS
  14. HOME_IS_B2B
  15. AWAY_IS_B2B
  16. DIFF_ELO
  17. DIFF_eFG_PCT
  18. DIFF_TO_RATIO
  19. DIFF_FT_RATE
  20. DIFF_OREB_PCT
  21. DIFF_PTS_ALLOWED ⭐ MỚI


In [15]:
# Kiểm tra không NaN, không inf
v2_df = merged[feature_set_v2]
print('Kiểm tra chất lượng features V2:')
print(f'  NaN:  {v2_df.isnull().sum().sum()}')
print(f'  Inf:  {np.isinf(v2_df).sum().sum()}')
print(f'  Shape: {v2_df.shape}')

# Correlation của 4 features mới với HOME_WIN
print(f'\nCorrelation với HOME_WIN:')
print('-' * 40)
for feat in feature_set_v2:
    corr = merged[feat].corr(merged['HOME_WIN'])
    marker = ' ⭐' if feat in DIFF_FEATURES_V2.values() else ''
    print(f'  {feat:22s}: {corr:+.4f}{marker}')

Kiểm tra chất lượng features V2:
  NaN:  0
  Inf:  0
  Shape: (6061, 20)

Correlation với HOME_WIN:
----------------------------------------
  DIFF_PTS              : +0.1514
  DIFF_FG_PCT           : +0.1511
  DIFF_FG3_PCT          : +0.1033
  DIFF_FT_PCT           : +0.0666
  DIFF_OREB             : -0.0214
  DIFF_DREB             : +0.1123
  DIFF_AST              : +0.0759
  DIFF_STL              : +0.0432
  DIFF_BLK              : +0.0610
  DIFF_TOV              : -0.1120
  DIFF_WIN_PCT          : +0.2931
  DIFF_WIN_STREAK       : +0.1932
  DIFF_REST_DAYS        : +0.0575
  HOME_IS_B2B           : -0.0572
  AWAY_IS_B2B           : +0.0341
  DIFF_ELO              : +0.3417 ⭐
  DIFF_eFG_PCT          : +0.1627 ⭐
  DIFF_TO_RATIO         : -0.1018 ⭐
  DIFF_FT_RATE          : +0.0479 ⭐
  DIFF_OREB_PCT         : +0.0015 ⭐


# 8. Train/Test split (2 lần chia)

In [16]:
# =============================================
# LẦN 1: Đánh giá (báo cáo)
# =============================================
eval_train_seasons = ['2021-22', '2022-23', '2023-24', '2024-25']
eval_test_season = '2025-26'

eval_train = merged[merged['SEASON'].isin(eval_train_seasons)].copy()
eval_test = merged[merged['SEASON'] == eval_test_season].copy()

print('LẦN 1 — ĐÁNH GIÁ MODEL (BÁO CÁO)')
print('=' * 50)
print(f'Train: {len(eval_train)} trận | {eval_train["GAME_DATE"].min()} → {eval_train["GAME_DATE"].max()}')
print(f'Test:  {len(eval_test)} trận | {eval_test["GAME_DATE"].min()} → {eval_test["GAME_DATE"].max()}')

assert eval_train['GAME_DATE'].max() < eval_test['GAME_DATE'].min(), '⚠️ Data leakage!'
print(f'Train HOME_WIN: {eval_train["HOME_WIN"].mean():.1%} | Test HOME_WIN: {eval_test["HOME_WIN"].mean():.1%}')

LẦN 1 — ĐÁNH GIÁ MODEL (BÁO CÁO)
Train: 4852 trận | 2021-10-22 00:00:00 → 2025-04-13 00:00:00
Test:  1209 trận | 2025-10-24 00:00:00 → 2026-04-12 00:00:00
Train HOME_WIN: 55.4% | Test HOME_WIN: 55.3%


In [ ]:
final_train = merged.copy()

print('LẦN 2 — MODEL CUỐI CÙNG (DEMO)')
print('=' * 50)
print(f'Train: {len(final_train)} trận (TẤT CẢ dữ liệu)')
print(f'Thời gian: {final_train["GAME_DATE"].min()} → {final_train["GAME_DATE"].max()}')
print(f'→ Predict: các trận SAU {final_train["GAME_DATE"].max()}')

LẦN 2 — MODEL CUỐI CÙNG (DEMO)
Train: 6061 trận (TẤT CẢ dữ liệu)
Thời gian: 2021-10-22 00:00:00 → 2026-04-12 00:00:00
→ Predict: các trận SAU 2026-04-12 00:00:00


In [18]:
# Tách X, y
LABEL = 'HOME_WIN'

# Lần 1 — Đánh giá
X_eval_train = eval_train[feature_set_v2]
X_eval_test  = eval_test[feature_set_v2]
y_eval_train = eval_train[LABEL]
y_eval_test  = eval_test[LABEL]

# Lần 2 — Demo
X_final = final_train[feature_set_v2]
y_final = final_train[LABEL]

print('SHAPES:')
print(f'  Lần 1: train {X_eval_train.shape}, test {X_eval_test.shape}')
print(f'  Lần 2: {X_final.shape}')
print(f'  Features: {len(feature_set_v2)} cột')
print(f'  Label: {LABEL}')

SHAPES:
  Lần 1: train (4852, 20), test (1209, 20)
  Lần 2: (6061, 20)
  Features: 20 cột
  Label: HOME_WIN


# 9. Lưu dataset V3

In [ ]:
import os
import json
from google.colab import files 

save_dir = 'output'
os.makedirs(save_dir, exist_ok=True)

# 1. Lưu toàn bộ merged
merged_path = f'{save_dir}/nba_model_ready_v3.csv'
merged.to_csv(merged_path, index=False)

# 2. Lưu train/test splits cho V3
eval_train = merged[merged['SEASON'].isin(eval_train_seasons)].copy()
eval_test  = merged[merged['SEASON'] == eval_test_season].copy()

train_path = f'{save_dir}/eval_train_v3.csv'
test_path = f'{save_dir}/eval_test_v3.csv'

eval_train.to_csv(train_path, index=False)
eval_test.to_csv(test_path, index=False)

print(f'✅ Đã lưu 3 file CSV vào thư mục: {save_dir}/')

# 3. Lưu config V3
feature_config_v3 = {
    'version': 'v3',
    'feature_set_v2': feature_set_v2,
    'feature_set_v3': feature_set_v3,
    'target_regression': 'MARGIN',
    'target_classification': 'HOME_WIN',
    'ema_features_all': EMA_FEATURES_ALL,
    'ema_span': EMA_SPAN,
    'diff_features_v1': DIFF_FEATURES_V1,
    'diff_features_v2': DIFF_FEATURES_V2,
    'team_features': team_features,
    'eval_train_seasons': eval_train_seasons,
    'eval_test_season': eval_test_season,
    'elo_params': {
        'K': K,
        'home_advantage': HOME_ADVANTAGE,
        'initial_elo': INITIAL_ELO,
        'season_carryover': SEASON_CARRYOVER,
    },
    'changes_from_v2': [
        'Added raw HOME_PTS, AWAY_PTS, MARGIN as target',
        'Added DIFF_PTS_ALLOWED as feature (points allowed difference)',
        'Total features: 21 (V2:20 + DIFF_PTS_ALLOWED)'
    ],
}

config_path = f'{save_dir}/feature_config_v3.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(feature_config_v3, f, indent=2, ensure_ascii=False)

print(f'✅ Đã lưu file config vào: {config_path}')


print("\nĐang chuẩn bị tải file về máy...")
files_to_download = [merged_path, train_path, test_path, config_path]

for file_path in files_to_download:
    files.download(file_path) # Kích hoạt lệnh tải xuống của trình duyệt

print("🚀 Hãy kiểm tra thư mục Download trên máy tính của bạn!")

✅ Đã lưu 3 file CSV vào thư mục: output/
✅ Đã lưu file config vào: output/feature_config_v3.json

Đang chuẩn bị tải file về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Hãy kiểm tra thư mục Download trên máy tính của bạn!
